In [0]:
# ═══════════════════════════════════════════════
# 03_GOLD — Build 3 business-ready Gold tables
# ═══════════════════════════════════════════════
from pyspark.sql.functions import year, date_format, avg, round as spark_round

silver = spark.table("silver_card_spending")

# Gold 1: Yearly trend
(silver
    .groupBy(year("date").alias("year"), "category")
    .agg(spark_round(avg("spend_index"), 2).alias("avg_spend_index"))
    .orderBy("year", "category")
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("gold_spend_by_year"))
print("gold_spend_by_year written.")

# Gold 2: Seasonal pattern (same calendar day across all years)
(silver
    .groupBy(date_format("date", "MM-dd").alias("month_day"), "category")
    .agg(spark_round(avg("spend_index"), 2).alias("avg_spend_index"))
    .orderBy("month_day", "category")
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("gold_spend_seasonal"))
print("gold_spend_seasonal written.")

# Gold 3: Daily time series
(silver
    .select("date", "category", "spend_index")
    .orderBy("date", "category")
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("gold_spend_daily"))
print("gold_spend_daily written.")

gold_spend_by_year written.
gold_spend_seasonal written.
gold_spend_daily written.
